# 02 Quality Check

이 노트북은 품질 필터링 결과를 확인하기 위한 노트북입니다.

목적:
- `quality_filtered_metadata.parquet` 결과 확인
- 탈락 이미지 확인
- 블러 점수 분포 확인
- 실제 이미지 샘플 시각화


In [ ]:

from pathlib import Path
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

ROOT = Path("..").resolve()

QUALITY_METADATA_PATH = ROOT / "data" / "metadata" / "quality_filtered_metadata.parquet"
QUALITY_REPORT_PATH = ROOT / "outputs" / "reports" / "quality_filter" / "quality_filter_summary.json"
REJECTED_PATH = ROOT / "outputs" / "reports" / "quality_filter" / "rejected_images.csv"

QUALITY_METADATA_PATH


In [ ]:

quality_df = pd.read_parquet(QUALITY_METADATA_PATH)
print("quality_df shape:", quality_df.shape)
quality_df.head()


In [ ]:

if QUALITY_REPORT_PATH.exists():
    with open(QUALITY_REPORT_PATH, "r", encoding="utf-8") as f:
        summary = json.load(f)
    summary
else:
    print("quality_filter_summary.json does not exist.")


In [ ]:

quality_cols = [
    "original_food_name",
    "business_category",
    "product_group",
    "quality_pass",
    "quality_status",
    "quality_score",
    "blur_score",
    "actual_width",
    "actual_height",
    "image_path",
]

existing_cols = [col for col in quality_cols if col in quality_df.columns]
quality_df[existing_cols].head(30)


In [ ]:

if "quality_status" in quality_df.columns:
    status_dist = quality_df["quality_status"].value_counts().reset_index()
    status_dist.columns = ["quality_status", "count"]
    status_dist["ratio"] = status_dist["count"] / len(quality_df)
    display(status_dist)


In [ ]:

if "quality_score" in quality_df.columns:
    plt.figure(figsize=(8, 5))
    plt.hist(quality_df["quality_score"].dropna(), bins=20)
    plt.title("Quality Score Distribution")
    plt.xlabel("Quality Score")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()


In [ ]:

if "blur_score" in quality_df.columns:
    plt.figure(figsize=(8, 5))
    plt.hist(quality_df["blur_score"].dropna(), bins=50)
    plt.title("Blur Score Distribution")
    plt.xlabel("Blur Score")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()


In [ ]:

if REJECTED_PATH.exists():
    rejected_df = pd.read_csv(REJECTED_PATH)
    print("rejected_df shape:", rejected_df.shape)
    display(rejected_df.head(50))
else:
    rejected_df = pd.DataFrame()
    print("No rejected_images.csv found.")


In [ ]:

def show_image_grid(df, image_col="image_path", title_col="original_food_name", n=12, cols=4):
    sample = df.head(n).copy()
    rows = int(np.ceil(len(sample) / cols))

    plt.figure(figsize=(cols * 4, rows * 4))

    for i, (_, row) in enumerate(sample.iterrows(), start=1):
        image_path = Path(str(row[image_col]))

        plt.subplot(rows, cols, i)
        plt.axis("off")

        try:
            img = Image.open(image_path).convert("RGB")
            plt.imshow(img)
            plt.title(str(row.get(title_col, ""))[:30])
        except Exception as e:
            plt.title(f"error: {e}")

    plt.tight_layout()
    plt.show()


In [ ]:

sample_df = quality_df.sample(min(12, len(quality_df)), random_state=42)
show_image_grid(sample_df, n=12, cols=4)


In [ ]:

if len(rejected_df) > 0 and "image_path" in rejected_df.columns:
    show_image_grid(rejected_df, n=12, cols=4)
else:
    print("No rejected images to display.")


In [ ]:

if "business_category" in quality_df.columns:
    business_quality = (
        quality_df
        .groupby("business_category")
        .agg(
            count=("business_category", "size"),
            avg_quality_score=("quality_score", "mean"),
            avg_blur_score=("blur_score", "mean"),
            min_width=("actual_width", "min"),
            min_height=("actual_height", "min"),
        )
        .reset_index()
        .sort_values("count", ascending=False)
    )
    display(business_quality)
